## **Import Required libraries**

In [0]:
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql.functions import explode,col,current_date,upper,when,trim,lit,from_json,year, min, max, month,broadcast

## **Get Current Date**

In [0]:
today = datetime.today()
current_year = today.strftime("%Y")
current_month = today.strftime("%m")
date_str = today.strftime("%Y-%m-%d")

## **Path - Source**

In [0]:
path_movie = f"/Volumes/movie_project/bronze/movies_extended/{current_year}/{current_month}/movies_details_{date_str}.json"
path_cast_crew = f"/Volumes/movie_project/bronze/credits/{current_year}/{current_month}/credits_{date_str}.json"

## **Path - Sink**

In [0]:
genre = '/Volumes/movie_project/silver/genre'
country = '/Volumes/movie_project/silver/country'
prod_country = '/Volumes/movie_project/silver/prod_country'
prod_company = '/Volumes/movie_project/silver/prod_company'
spoken_language = '/Volumes/movie_project/silver/spoken_language'
movie = '/Volumes/movie_project/silver/movie'
cast = '/Volumes/movie_project/silver/cast'
crew = '/Volumes/movie_project/silver/crew'

## **Read Data**

In [0]:
#Read

df = spark.read.format('json').option('multiline','true').load(path_movie) #need to persist

In [0]:
df = df.withColumn(
        "release_date",
        when(col("release_date").isNull() | (col("release_date") == ""), lit("1900-01-01"))
        .otherwise(col("release_date"))
    ) \
    .withColumn("release_date", col("release_date").cast("date")) \
    .withColumn("release_year", year(col("release_date")))\
    .withColumn("release_month",month(col("release_date")))

In [0]:
a = df.select('release_month').distinct().collect()
b = df.select('release_year').distinct().collect()

In [0]:
month_list = []
year_list = []

for i in a:
    month_list.append(i.release_month)
    print(i.release_month)
for i in b:
    year_list.append(i.release_year)
    print(i.release_year)


1
2
2024


In [0]:
# Set values
dbutils.jobs.taskValues.set(key="month_list", value=month_list)
dbutils.jobs.taskValues.set(key="year_list", value=year_list)

## **Genre**

In [0]:
#Transform 

df_genre = df.select(col('id').alias('movie_id'),'genres','release_year','release_month')\
            .withColumn('genres',explode('genres'))\
            .select('movie_id','genres.id','genres.name','release_year','release_month')\
            .dropDuplicates(['movie_id','id'])\
            .withColumn('modified_date_silver',current_date())

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,genre):

    delta_table = DeltaTable.forPath(spark, genre)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_genre.alias('s'),
        ((col('s.movie_id') == col('t.movie_id')) & (col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_genre.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',genre)\
        .save()



## **Country**

In [0]:
#Transform

df_country = df.select(col('id').alias('movie_id'),'origin_country','release_year','release_month')\
                .withColumn('origin_country',explode('origin_country'))\
                .withColumn('origin_country',upper('origin_country'))\
                .dropDuplicates(['movie_id','origin_country'])\
                .withColumn('modified_date_silver',current_date())


In [0]:
# Load

if DeltaTable.isDeltaTable(spark,country):

    delta_table = DeltaTable.forPath(spark, country)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_country.alias('s'),
        ((col('s.movie_id') == col('t.movie_id')) & (col('s.origin_country') == col('t.origin_country')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll().execute()

else:
    # Initial Load
    df_country.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',country)\
        .save()

## **Production_countries**

In [0]:
#Transform

df_prod_country = df.select(col('id').alias('movie_id'),'production_countries','release_year',      'release_month')\
                    .withColumn('production_countries',explode('production_countries'))\
                    .select('movie_id',col('production_countries.iso_3166_1').alias('id'),'production_countries.name','release_year','release_month')\
                    .dropDuplicates(['movie_id','id'])\
                    .withColumn('modified_date_silver',current_date())

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,prod_country):

    delta_table = DeltaTable.forPath(spark, prod_country)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_prod_country.alias('s'),
       ((col('s.movie_id') == col('t.movie_id')) & (col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll().execute()

else:
    # Initial Load
    df_prod_country.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',prod_country)\
        .save()

## **Production_companies**

In [0]:
#Transform

df_prod_company = df.select(col('id').alias('movie_id'),'production_companies','release_year','release_month')\
                    .withColumn('production_companies',explode('production_companies'))\
                    .select('movie_id','production_companies.id','production_companies.name',
                            'production_companies.origin_country','release_year','release_month')\
                    .withColumn('origin_country',when((trim(col('origin_country'))=='') | (col('origin_country').isNull()) ,lit('Unknown')).otherwise(col('origin_country')))\
                    .dropDuplicates(['movie_id','id'])\
                    .withColumn('modified_date_silver',current_date())

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,prod_company):

    delta_table = DeltaTable.forPath(spark, prod_company)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_prod_company.alias('s'),
       ((col('s.movie_id') == col('t.movie_id')) & (col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    # Initial Load
    df_prod_company.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',prod_company)\
        .save()

## **Spoken_languages**

In [0]:
#Transform

df_spoken_languages = df.select(col('id').alias('movie_id'),'spoken_languages','release_year','release_month')\
                        .withColumn('spoken_languages',explode('spoken_languages'))\
                        .select('movie_id',col('spoken_languages.iso_639_1').alias('id'),'spoken_languages.name','spoken_languages.english_name','release_year','release_month')\
                        .withColumn('id',upper('id'))\
                        .dropDuplicates(['movie_id','id'])\
                        .withColumn('modified_date_silver',current_date())
                        

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,spoken_language):

    delta_table = DeltaTable.forPath(spark, spoken_language)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_spoken_languages.alias('s'),
       ((col('s.movie_id') == col('t.movie_id')) & (col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll().execute()

else:
    # Initial Load
    df_spoken_languages.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',spoken_language)\
        .save()

## **Movie**

In [0]:
#Transformation

df_movie = df.drop('belongs_to_collection','genres','origin_country','production_companies','production_countries','spoken_languages')\
            .withColumn('is_adult',when(col('adult')==True,lit('Yes')).otherwise(lit('No')))\
            .withColumn('homepage',when(trim(col('homepage'))=='',lit('no path is available')).otherwise(col('homepage')))\
            .withColumn('tagline',when(trim(col('tagline'))=='',lit('no slogan is available')).otherwise(col('tagline')))\
            .withColumn('overview',when(trim(col('overview'))=='',lit('no description is available')).otherwise(col('overview')))\
            .withColumn('original_language',upper('original_language'))\
            .fillna('no path is available',subset=['backdrop_path','poster_path'])\
            .fillna('ttno_id',subset=['imdb_id'])\
            .dropDuplicates(['id'])\
            .withColumn('modified_date_silver',current_date())
            
            
            

            

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,movie):

    delta_table = DeltaTable.forPath(spark, movie)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_movie.alias('s'),
       ((col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    # Initial Load
    df_movie.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',movie)\
        .save()

## **Cast**

In [0]:
df = spark.read.format('json').option('multiline','true').load(path_cast_crew)

In [0]:
#Transform

df_cast = df.select(col('id').alias('movie_id'),'cast')\
            .withColumn('cast',explode('cast'))\
            .select('movie_id','cast.*')\
            .withColumn('character',when(trim(col('character'))=='','not specified').otherwise(col('character')))\
            .withColumn('gender',col('gender').cast('string'))\
            .withColumn('gender',when(col('gender')==0,lit('Not specified')).when(col('gender')==1,lit('Female')).when(col('gender')==2,lit('Male')).when(col('gender')==3,lit('Non-binary')).otherwise(col('gender')))\
            .fillna('no path is available',subset=['profile_path'])\
            .dropDuplicates(['movie_id','credit_id'])\
            .withColumn('modified_date_silver',current_date())

In [0]:
df_date = df_movie.select(col('id').alias('movie_id'),'release_year','release_month')

df_cast = df_cast.join(broadcast(df_date), on = 'movie_id',how ='left')

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,cast):

    delta_table = DeltaTable.forPath(spark, cast)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_cast.alias('s'),
       ((col('s.movie_id') == col('t.movie_id'))&(col('s.credit_id') == col('t.credit_id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    # Initial Load
    df_cast.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',cast)\
        .save()

## **Crew**

In [0]:
#Transfrom

df_crew = df.select(col('id').alias('movie_id'),'crew')\
            .withColumn('crew',explode('crew'))\
            .select('movie_id','crew.*')\
            .withColumn('gender',col('gender').cast('string'))\
            .withColumn('gender',when(col('gender')==0,lit('Not specified')).when(col('gender')==1,lit('Female')).when(col('gender')==2,lit('Male')).when(col('gender')==3,lit('Non-binary')).otherwise(col('gender')))\
            .fillna('no path is available',subset=['profile_path'])\
            .dropDuplicates(['movie_id','credit_id'])\
            .withColumn('modified_date_silver',current_date())

In [0]:
df_crew = df_crew.join(broadcast(df_date), on = 'movie_id',how ='left')

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,crew):

    delta_table = DeltaTable.forPath(spark, crew)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_crew.alias('s'),
       ((col('s.movie_id') == col('t.movie_id'))&(col('s.credit_id') == col('t.credit_id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

else:
    # Initial Load
    df_crew.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',crew)\
        .save()